In [13]:
from pickle import STRING

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.widgets import Lasso
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge ,Lasso,ElasticNet,LassoCV,RidgeCV,ElasticNetCV
from sklearn.metrics import r2_score , mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures


In [14]:
df=pd.read_csv("insurance.csv")
df.head()


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [15]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [16]:

df["sex"] = (df["sex"] == "female").astype(int)

In [17]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [18]:
df[df.isnull().any(axis=1)]

,age,sex,bmi,children,smoker,region,charges


In [19]:
( df["smoker"] == 'yes').astype(int)

0       1
1       0
2       0
3       0
4       0
       ..
1333    0
1334    0
1335    0
1336    0
1337    1
Name: smoker, Length: 1338, dtype: int64

In [20]:
df["smoker"]= ( df["smoker"] == 'yes').astype(int)

In [21]:
#encoding
df=pd.get_dummies(df,columns=["region"],drop_first=True)

In [22]:
df.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,1,27.900,0,1,16884.92400,False,False,True
1,18,0,33.770,1,0,1725.55230,False,True,False
2,28,0,33.000,3,0,4449.46200,False,True,False
3,33,0,22.705,0,0,21984.47061,True,False,False
4,32,0,28.880,0,0,3866.85520,True,False,False


In [23]:
df.shape

(1338, 9)

In [24]:
df.value_counts()

age  sex  bmi     children  smoker  charges      region_northwest  region_southeast  region_southwest
19   0    30.590  0         0       1639.56310   True              False             False               2
18   0    33.535  0         1       34617.84065  False             False             False               1
64   0    36.960  2         1       49577.66240  False             True              False               1
          34.500  0         0       13822.80300  False             False             True                1
18   0    23.750  0         0       1705.62450   False             False             False               1
                                                                                                        ..
64   0    24.700  1         0       30166.61817  True              False             False               1
          25.600  2         0       14988.43200  False             False             True                1
          26.410  0         0       14394.

In [34]:
df.isnull().sum()

age                 0
sex                 0
bmi                 0
children            0
smoker              0
charges             0
region_northwest    0
region_southeast    0
region_southwest    0
dtype: int64

In [25]:
df.describe()

,age,sex,bmi,children,smoker,charges
count,1338.000000,1338.000000,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,0.494768,30.663397,1.094918,0.204783,13270.422265
std,14.049960,0.500160,6.098187,1.205493,0.403694,12110.011237
min,18.000000,0.000000,15.960000,0.000000,0.000000,1121.873900
25%,27.000000,0.000000,26.296250,0.000000,0.000000,4740.287150
50%,39.000000,0.000000,30.400000,1.000000,0.000000,9382.033000
75%,51.000000,1.000000,34.693750,2.000000,0.000000,16639.912515
max,64.000000,1.000000,53.130000,5.000000,1.000000,63770.428010


In [26]:
#train and test

X = df.drop("charges", axis=1)
y = df["charges"]

X_train , X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=15)


In [27]:
#standartization
scaler=StandardScaler()
X_train_scaled= scaler.fit_transform(X_train)
X_test_scaled= scaler.transform(X_test)

In [28]:
#linearregression
regression=LinearRegression()
regression.fit(X_train_scaled,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [29]:
y_pred=regression.predict(X_test_scaled)
r2_score(y_test,y_pred)

0.7709928565663493

In [30]:
Lasso, LassoCV = Lasso(), LassoCV()
Ridge, RidgeCV = Ridge(), RidgeCV()
ElasticNet, ElasticNetCV = ElasticNet(), ElasticNetCV()

In [31]:
models = (regression, Lasso, LassoCV, Ridge, RidgeCV, ElasticNet, ElasticNetCV)

In [32]:
#pipeline
for model in models:
    pipe=Pipeline(
        [
          ("Scaler" ,StandardScaler()),
            ("model", model)
        ]
    )
    pipe.fit(X_train,y_train)
    score=pipe.score(X_test,y_test)
    print(f"Model : {model.__class__.__name__}\n\n" +
          f"R2 Score : {score}\n")

    print(f"Model Coefficients : {model.coef_}")

    print("=" * 50)


Model : LinearRegression

R2 Score : 0.7709928565663493

Model Coefficients : [3692.05530173  -21.26551004 2082.90357967  731.15551483 9682.92328646
 -168.34235204 -474.63515837 -420.50865902]
Model : Lasso

R2 Score : 0.7710205381848835

Model Coefficients : [3691.21684271  -20.37754714 2081.52465805  730.19086438 9682.00847259
 -165.30891268 -471.06223848 -417.33026864]
Model : LassoCV

R2 Score : 0.7719455182845418

Model Coefficients : [3652.86058641   -0.         2016.24981422  685.13167779 9637.24914315
  -24.7591638  -305.44413979 -269.59842723]
Model : Ridge

R2 Score : 0.7710251514366444

Model Coefficients : [3688.55625122  -22.11301666 2080.91750997  730.88248899 9673.57238671
 -167.84141327 -472.46811609 -419.64612132]
Model : RidgeCV

R2 Score : 0.7710251514366535

Model Coefficients : [3688.55625122  -22.11301666 2080.91750996  730.88248898 9673.57238671
 -167.84141327 -472.46811609 -419.64612132]
Model : ElasticNet

R2 Score : 0.688052226357778

Model Coefficients : [244

In [33]:
#polynomial features
poly=PolynomialFeatures(degree=3)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

model = LinearRegression()
model.fit(X_train_poly, y_train)

y_pred = model.predict(X_test_poly)
r2_score(y_test,y_pred)

0.8362454201982794